<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Rapid Application Development<br>using Large Language Models</b></h1>
<h2><b>Notebook 6.5: [Advanced]</b> Running A GenAI Server</h2>
<br>

This notebook complements **Notebook 6** by providing a dedicated notebook - and hence a dedicated event loop - to kickstart a series of GenAI microservices. This notebook will enable the local model APIs that will be used through the rest of the course, and features some more advanced tangents for those interested. 

As a prelude to **multi-GPU scaled deployment**, this notebook provides an oversimplified but functionally-sufficient example of an **on-device deployment workflow** which can be used to offer a standardized API for a consumer-grade or on-device deployment strategy. Since this course uses a single GPU instance and has many notebooks, you will run into problems if any one notebook overallocates model resources or fails to clear cache. As such, this is your chance to deploy some smaller models once and call them as necessary. 

> **NOTE:** This notebook is an advanced tangent, should be ran to facilitate some later feature sets, and can be optionally understood if you have the time and interest. **Do not feel pressured to understand everything in this notebook and halt your progress because of it!**

> **NOTE:** Before running this notebook, we recommend shutting down all active kernel instances from notebooks 5 and below. This notebook is intended to commandeer the majority of the available GPU memory to launch multiple models. 

<hr>
<br>

## **Part 1:** Making A Custom Image Server with FastAPI

To start, we can make a server to wrap [FLUX.2-klein-4B](https://build.nvidia.com/black-forest-labs/flux_2-klein-4b), using the same diffusion API pattern you experimented with at the end of Notebook 5. This is intended as an example application derived largely from the HuggingFace documented example ([narrative](https://huggingface.co/docs/diffusers/main/en/using-diffusers/create_a_server) and [code](https://github.com/huggingface/diffusers/blob/main/examples/server/server.py)), and provides a peek behind the curtain of inference server deployments without exploring too much of the complexity. 

**Warning:** FastAPI and server development is not within the scope of this class, and serves as a simplified example for those interested. Feel free to run the service as-is and assume the API will roughly work and operate reliably enough to fulfill the exercises, but we do not recommend using this solution for production. If you are interested in productionalizing an image generation server, consider using:
- [NVIDIA image-model offerings](https://build.nvidia.com/explore/discover) for self-hosted options.
- [Automatic1111-style](https://github.com/NVIDIA/Stable-Diffusion-WebUI-TensorRT/tree/main) and [ComfyUI-style](https://github.com/comfyanonymous/ComfyUI_TensorRT) Stable-Diffusion WebUIs for streamlined accelerated deployments. 
- Hosted APIs like those offered by OpenAI and StabilityAI for non-self-hosted options.
- Inference server projects like [vLLM](https://github.com/vllm-project/vllm) for inspirations on productionalized deployments. 

#### **Note The Following Features:**
- **This is sufficient for a background server that mimics OpenAI's Dalle API for a single-user use-case.**
- It roughly follows OpenAI's Dalle API with minimal functionality of generating a novel image.
- Once the server is running, other services can call it via a port interface without having to reload or re-construct the resource allocated in the server. 

#### **Note The Following Critical Limitations:**
- **A pipeline cannot safely run concurrent requests.** The server serializes generation and produces one image at a time to keep memory use bounded. This remains a single-user example; a production service also needs admission control and capacity planning. 
- **Model discovery is minimal.** `/v1/models` names the model, but callers still need the request schema below to understand supported sizes and options.
- **The server saves images directly to the shared notebook filesystem.** This is simplified for convenience, and is sufficient for meaningful interactions in a local deployment context.
    - [OpenAI](https://platform.openai.com/docs/guides/images) would save such an image to a temporary publicly-accessible address. The caller can download the image and operate on it as usual.
    - [NVIDIA NIM microservices](https://docs.api.nvidia.com/nim/reference/stabilityai-stable-diffusion-xl-infer), and thereby `build.nvidia.com`, returns a buffer in its response which can be decoded and interpretted as an image by the caller.

In [1]:
%%writefile sdxl_app.py
"""Single-GPU course image service. Responses contain shared notebook filesystem paths."""
import asyncio
import gc
import logging
import os
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Literal
from uuid import uuid4

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator

MODEL_ID = 'black-forest-labs/FLUX.2-klein-4B'
REVISION = 'e7b7dc27f91deacad38e78976d1f2b499d76a294'
# Relative paths resolve in both the lab and the separately mounted grader.
IMAGE_DIR = Path(os.getenv('DLI_IMAGE_DIR', 'generated_images'))
logger = logging.getLogger(__name__)


#########################################################################################
## Lifespan: load the pipeline, warm up GPU allocations, and clean up on shutdown
#########################################################################################

@asynccontextmanager
async def lifespan(app):
    import torch
    from diffusers import Flux2KleinPipeline
    if not torch.cuda.is_available():
        raise RuntimeError('FLUX requires a CUDA GPU. Start this service in the course GPU lab.')
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    app.state.pipeline = None
    app.state.lock = asyncio.Lock()
    try:
        pipeline = Flux2KleinPipeline.from_pretrained(
            MODEL_ID, revision=REVISION, torch_dtype=torch.bfloat16,
        )
        # Offload inactive components so the image model and VLM share one GPU.
        pipeline.enable_model_cpu_offload()
        pipeline.set_progress_bar_config(disable=True)
        # Allocate inference buffers before reporting readiness.
        pipeline(prompt='A blank test image', width=512, height=512, num_inference_steps=4, guidance_scale=1.0,
                 generator=torch.Generator('cuda').manual_seed(0))
        app.state.pipeline = pipeline
        yield
    finally:
        app.state.pipeline = None
        if 'pipeline' in locals():
            del pipeline
        gc.collect()
        torch.cuda.empty_cache()


app = FastAPI(lifespan=lifespan)


#########################################################################################
## Request schema: define the API and bound the work that fits this shared GPU
#########################################################################################

class ImageRequest(BaseModel):
    model: Literal['black-forest-labs/FLUX.2-klein-4B']
    prompt: str = Field(min_length=1, max_length=4000)
    size: str = '1024x1024'
    n: int = Field(default=1, ge=1, le=4)
    seed: int | None = Field(default=None, ge=0, le=2**32 - 1)

    @field_validator('size')
    @classmethod
    def valid_size(cls, value):
        try:
            width, height = map(int, value.lower().split('x'))
        except ValueError as error:
            raise ValueError('Use WIDTHxHEIGHT, for example 512x512.') from error
        if any(side < 512 or side > 1024 or side % 64 for side in (width, height)):
            raise ValueError('Each side must be 512..1024 pixels and divisible by 64.')
        return f'{width}x{height}'


@app.get('/health')
@app.get('/')
async def health():
    ready = getattr(app.state, 'pipeline', None) is not None
    if not ready:
        raise HTTPException(503, 'FLUX is not ready.')
    return {'status': 'ready', 'model': MODEL_ID, 'revision': REVISION}


@app.get('/v1/models')
async def models():
    return {'object': 'list', 'data': [{'id': MODEL_ID, 'object': 'model'}]}


#########################################################################################
## Inference: generate images, retain their files, and return paths to the caller
#########################################################################################

def render(request):
    import torch
    width, height = map(int, request.size.split('x'))
    paths = []
    try:
        # Generate one image at a time so n does not multiply peak GPU memory.
        for index in range(request.n):
            generator = torch.Generator('cuda')
            generator.manual_seed(request.seed + index) if request.seed is not None else generator.seed()
            image = app.state.pipeline(
                prompt=request.prompt, width=width, height=height, num_inference_steps=4, guidance_scale=1.0,
                generator=generator,
            ).images[0]
            target = IMAGE_DIR / f'draw_{uuid4().hex}.png'
            image.save(target)
            paths.append(target)
        return {'data': [{'url': str(path)} for path in paths]}
    except BaseException:
        for path in paths:
            path.unlink(missing_ok=True)
        raise


#########################################################################################
## Endpoint: serialize pipeline access while keeping the event loop responsive
#########################################################################################

@app.post('/v1/images/generations')
async def generate(request: ImageRequest):
    await health()
    try:
        await asyncio.wait_for(app.state.lock.acquire(), timeout=60)
    except asyncio.TimeoutError:
        raise HTTPException(429, 'Image generation is busy; retry after the current request finishes.')
    task = asyncio.create_task(asyncio.to_thread(render, request))
    try:
        return await asyncio.shield(task)
    except asyncio.CancelledError:
        # A running CUDA call cannot be canceled by abandoning its Python thread.
        # Hold the lock until it finishes to prevent concurrent pipeline mutation.
        try:
            await task
        except Exception:
            pass
        raise
    except Exception as error:
        logger.error('Image generation failed (%s)', type(error).__name__)
        raise HTTPException(503, 'Image generation failed. Inspect the FLUX service log.') from None
    finally:
        app.state.lock.release()


Writing sdxl_app.py


<hr>
<br>

## **Part 2:** Getting Pre-Made Standardized Servers Using vLLM

For a higher layer of abstraction, many people like to use inference services like [vLLM](https://github.com/vllm-project/vllm) to help deploy scalable and standardized LLM APIs. This framework automatically integrates with the Transformers model repository to download model weights and deserialize from config to pipeline. At the same time, it also offers abstractions for memory management, multi-user deployment, and API standards to wrap a resilient server around the loaded model resource. 

Using vLLM, we will deploy a small model that can run from within this notebook on our available resources: [NVIDIA-Nemotron-Nano-12B-v2-VL-BF16](https://huggingface.co/nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16).

This model, developed by NVIDIA for small language model use-cases, uses the multimodal fusion principles from Notebook 5 to perform joint reasoning with images and text. At the same time, it adopts the standard OpenAI-style API to largely interoperate with the likes of [**OpenAI's GPT-4o**](https://openai.com/index/hello-gpt-4o/), [**NVIDIA's NVLM**](https://arxiv.org/abs/2409.11402), and the open-sourced [**Llama 3.2 (2024)**](https://ai.meta.com/blog/llama-3-2-connect-2024-vision-edge-mobile-devices/).

To deploy an inference server, the following command will work in our environment:

```sh
vllm serve "nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16" \
  --revision ca9543b126e8bf3176916d3d305ccc415f89fd4d \
  --code-revision ca9543b126e8bf3176916d3d305ccc415f89fd4d \
  --max_model_len 8192 \
  --gpu-memory-utilization 0.625 \
  --limit-mm-per-prompt '{"image":1,"video":0}' \
  --enforce-eager \
  --port 9002 \
  --max-num-seqs 2 \
  --trust-remote-code
```

These values reserve room for the diffusion service used alongside the VLM. They are lab capacity
settings, not general production defaults. For production deployment, use a supported NVIDIA NIM
configuration and size it for the target workload.

<details>
<summary><b>Argument details</b></summary>

- **Revision:** fixes the downloaded model files to the reviewed Hugging Face commit.
- **Code revision:** fixes the model's custom NVIDIA implementation to its reviewed commit.
- **Port (9002):** exposes the local OpenAI-compatible endpoint.
- **Max model length (8192):** bounds context and KV-cache use below the model's maximum.
- **GPU memory utilization (0.625):** reserves about 28 GiB on the 48 GB L40S lab GPU. The launch cell calculates this fraction from the actual card capacity.
- **Multimodal limit:** permits one image per request and disables video to bound memory use.
- **Max number of sequences (2):** bounds concurrent sequence processing.
- **Enforce eager:** disables CUDA graph execution for this compatibility-focused lab setup.
- **Trust remote code:** loads the model's custom NVIDIA implementation from the pinned revision.

</details>


<hr>
<br>

## **Part 2.5:** BONUS: Wrangling These Servers With A Router

Reading more into the OpenAI API standard, you will notice the following server-level requirements: 
- You should be able to discover what the models accessible to you from the server.
- You should be able to communicate with the server in a standard and perdictable method.

The is very useful because you can build software around these endpoints that adjusts itself to the available model pool and invokes the endpoints in a predictable fashion. However, we currently have two different endpoints for our two LLM models. This is a bit of a hinderance for calling these endpoints, as we can see in this invocation example:

```python
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os 

## Saying Hello World to our model, ChatNVIDIA would auto-infer the name from discovery
vlm = ChatNVIDIA(model="nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16", base_url="http://127.0.0.1:9002/v1")
vlm.invoke("Describe a simple image of a city street.")
```

Wouldn't it be nicer if we could have them all aggregate under a single server?

```python
## But wouldn't it be nicer if we could have both models come from the same connector?
os.environ["NVIDIA_BASE_URL"] = "http://127.0.0.1:9004/v1"
chat = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b")
vlm = ChatNVIDIA(model="nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16")
```

Below, we define a basic router application which should help with this process. This is merely for convenience, but illustrates how one could aggregate multiple servers under one umbrella. This is, in fact, what `build.nvidia.com` does under the hood, and is a gateway to understand the `llm_client` microservice for those interested.

In [2]:
%%writefile router_app.py
import asyncio
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import Response, StreamingResponse, JSONResponse
import httpx
import json
import os
import traceback
from typing import Dict, Any, List, Tuple
from starlette.background import BackgroundTask

# --- Configuration and Defaults ---

DEFAULTS = {
    "TIMEOUT": 330,  # Request timeout in seconds for backend calls
    "URLS": "127.0.0.1:9002,llm_client:9000",  # Backend URLs
    "FILTER_KEYWORDS": "meta,mistral,nvidia,llama-3.2,huggingfacetb",  # Model filter keywords
    "CACHE_REFRESH_INTERVAL": 3600, # Auto-refresh interval in seconds (1 hour)
}

def get_var(key: str) -> str:
    """Fetch a configuration variable, falling back to defaults."""
    return os.getenv(key, DEFAULTS.get(key, ""))

def get_urls() -> List[str]:
    """Parse and normalize backend URLs."""
    # Ensure all URLs start with 'http://' or 'https://'
    return [url if "://" in url else f"http://{url.strip()}" for url in get_var("URLS").split(",")]

def get_filter_keywords() -> List[str]:
    """Parse filter keywords for model filtering."""
    return [kw.strip().lower() for kw in get_var("FILTER_KEYWORDS").split(",")]

HOP_HEADERS = {'connection', 'keep-alive', 'proxy-authenticate', 'proxy-authorization',
               'te', 'trailer', 'transfer-encoding', 'upgrade', 'content-length', 'content-encoding'}


def response_headers(response):
    return {key: value for key, value in response.headers.items() if key.lower() not in HOP_HEADERS}


# --- Utilities for Model Discovery ---

async def fetch_models_from_server(server_url: str) -> Dict[str, Any]:
    """Fetch available models from a single backend server."""
    timeout_sec = int(get_var("TIMEOUT"))
    try:
        # Use a fresh client for each request
        async with httpx.AsyncClient(timeout=15, trust_env=False) as client:
            response = await client.get(f"{server_url}/v1/models")
            if response.status_code == 200:
                entries = response.json().get("data", [])
                if not isinstance(entries, list) or any(not isinstance(row, dict) or not isinstance(row.get("id"), str) for row in entries):
                    raise ValueError("Invalid model catalog")
                return {"server": server_url, "models": entries}
            return {"error": f"Failed to fetch models from {server_url} (Status: {response.status_code})"}
    except Exception as e:
        # Log error but don't re-raise, discovery should be fault-tolerant
        return {"error": f"Model discovery failed: {type(e).__name__}"}

async def discover_models_and_cache(app: FastAPI) -> None:
    """Discover models from all backend servers and update the application cache."""
    print("--- Starting model discovery ---")

    tasks = [fetch_models_from_server(server) for server in get_urls()]
    results = await asyncio.gather(*tasks, return_exceptions=True)

    aggregated_models, errors = [], []
    for server, result in zip(get_urls(), results):
        if isinstance(result, dict) and "models" in result:
            # Tag models with their originating server
            for model in result["models"]:
                model["server"] = server
                aggregated_models.append(model)
        elif isinstance(result, dict) and "error" in result:
            errors.append(result["error"])
        else:
            errors.append(f"Unexpected result from {server}: {result}")

    # Apply filtering only to the discovered models
    keywords = get_filter_keywords()
    filtered_models = [
        model for model in aggregated_models
        if any(kw in model.get("id", "").lower() for kw in keywords)
    ]

    # Update cache
    app.state.models = filtered_models
    app.state.routes = {row["id"]: row["server"] for row in filtered_models}
    app.state.discovery_errors = errors
    print(f"--- Model discovery complete. Found {len(filtered_models)} models. Errors: {len(errors)} ---")

# --- Application Lifecycle and Caching ---

@asynccontextmanager
async def lifespan(app: FastAPI):
    """Manage application startup and shutdown events."""
    print(f"Configured backend servers: {get_urls()}")
    print(f"Model filter keywords: {get_filter_keywords()}")

    # Initialize state
    app.state.models: List[Dict[str, Any]] = []
    app.state.discovery_errors: List[str] = []
    app.state.timeout = int(get_var("TIMEOUT"))

    # Initial model discovery
    await discover_models_and_cache(app)

    # Setup periodic refresh (Optional, but good for dynamic backends)
    # Ref: https://fastapi.tiangolo.com/tutorial/events/
    # loop = asyncio.get_event_loop()
    # app.state.refresh_task = loop.create_task(
    #     periodic_model_refresh(app, int(get_var("CACHE_REFRESH_INTERVAL")))
    # )

    async with httpx.AsyncClient(timeout=httpx.Timeout(app.state.timeout, connect=10),
                                 trust_env=False, follow_redirects=False) as client:
        app.state.client = client
        yield # App is running

    # Cleanup on shutdown
    # app.state.refresh_task.cancel()
    print("Application shutdown complete.")

async def periodic_model_refresh(app: FastAPI, interval: int) -> None:
    """Periodically refresh the model cache."""
    while True:
        await asyncio.sleep(interval)
        print("--- Initiating periodic model cache refresh ---")
        await discover_models_and_cache(app)

# Initialize FastAPI application and apply lifespan
app = FastAPI(lifespan=lifespan)

# --- Endpoints ---

@app.get("/v1/models")
@app.get("/v1/models/")
async def list_models():
    """List all available (and filtered) models from the cache."""
    response = {"object": "list", "data": app.state.models}
    if app.state.discovery_errors:
        response["warnings"] = app.state.discovery_errors
    return JSONResponse(content=response)

@app.get("/v1/models/refresh")
async def refresh_models():
    """Trigger a manual refresh of the model cache."""
    await discover_models_and_cache(app)
    return JSONResponse(content={"status": "Model cache refreshed.", "count": len(app.state.models)})

async def refresh():
    await discover_models_and_cache(app)


@app.post('/v1/{operation:path}')
async def proxy(request: Request, operation: str):
    if operation not in {'chat/completions', 'completions', 'embeddings'}:
        raise HTTPException(404, 'Unsupported model operation.')
    try:
        body = await request.json()
    except ValueError:
        raise HTTPException(400, 'Invalid JSON request.')
    if not isinstance(body, dict) or not isinstance(body.get('model'), str):
        raise HTTPException(422, 'A model ID is required.')
    if not isinstance(body.get('stream', False), bool):
        raise HTTPException(422, 'stream must be a boolean.')
    model = body['model']
    if model not in app.state.routes:
        await refresh()
    target = app.state.routes.get(model)
    if target is None:
        raise HTTPException(404, 'Model is unavailable. Refresh /v1/models and inspect service readiness.')
    headers = {key: value for key, value in request.headers.items()
               if key.lower() in {'accept', 'content-type', 'cache-control'}}
    upstream = app.state.client.build_request('POST', target.rstrip('/') + '/v1/' + operation,
                                              json=body, headers=headers)
    try:
        response = await app.state.client.send(upstream, stream=True)
    except httpx.TimeoutException:
        raise HTTPException(504, 'The selected model service timed out.') from None
    except httpx.HTTPError:
        raise HTTPException(502, 'The selected model service could not be reached.') from None
    if not body.get('stream') or response.status_code >= 300:
        try:
            content = await response.aread()
            return Response(content, response.status_code, headers=response_headers(response))
        except httpx.TimeoutException:
            raise HTTPException(504, 'The selected model response timed out.') from None
        except httpx.HTTPError:
            raise HTTPException(502, 'The selected model response was interrupted.') from None
        finally:
            await response.aclose()

    async def chunks():
        try:
            async for chunk in response.aiter_bytes():
                yield chunk
        finally:
            await response.aclose()

    return StreamingResponse(chunks(), status_code=response.status_code,
                             headers=response_headers(response), background=BackgroundTask(response.aclose))


# --- Health Check ---

@app.get("/health")
async def health_check():
    """Simple health check endpoint."""
    required = {"nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16", "nvidia/nemotron-3.5-lightning-30b-a3b"}
    missing = sorted(required - set(app.state.routes))
    return JSONResponse({"status": "degraded" if missing else "ready", "missing_models": missing,
                         "discovery_errors": app.state.discovery_errors}, status_code=503 if missing else 200)

Writing router_app.py


<hr>
<br>

## **Part 3**: Kickstarting The Servers In Our Notebook

We've now explained how these servers are constructed, both behind the scenes and at a surface-level. **This is an advanced topic, so it's ok if this doesn't quite stick.** What's important is that you understand that:
- **It's possible to kickstart many LLM servers that can be used by one or many users.**
- **These servers can prevent resources from being allocated multiple times over.**
- **For good compatability, these servers usually follow a standard or easy-to-use API.**

We will need these features through the rest of the course, so please run the code cell below to kickstart your environment. This will kickstart each of your models, one at a time, until the VLM is available on port 9002, FLUX on 9003, and the router on 9004. Once again, this process is documented here for your own reference, and is not necessary to understand as part of the course learning objectives.

**NOTE:** 
- We recommend shutting down previous kernels to free up all of your resources for this action. 
- If you encounter an issue of freeze-up, please interrupt the kernel (square button) and try again.
- This command will produce a running log. For that reason, there is no text below the following cell. Please return back to **Notebook 6** once you're done. 

In [ ]:
import asyncio
import uvicorn
import subprocess
import nest_asyncio
from colorama import Fore, Style
import traceback
import shlex
import os
import signal
from composer.local_servers import gpu_budget

budget = gpu_budget()

# Allow nested asyncio event loops
nest_asyncio.apply()

async def run_cmd(cmd, label, color, start_event=None, end_event=None, end_trigger="http://0.0.0.0"):
    if start_event:
        await start_event.wait()  # Wait for the previous process to signal it's ready
    print(f"\nSTARTING {color}[{label}]{Style.RESET_ALL}")
    process = await asyncio.create_subprocess_exec(
        *shlex.split(cmd),
        start_new_session=True,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )
    async def read_stream(stream, color, label):
        while True:
            line = await stream.readline()
            if not line: break
            decoded_line = line.decode().strip()
            print(f"{color}[{label}]{Style.RESET_ALL} {decoded_line}")
            if end_event and end_trigger and end_trigger in decoded_line:
                end_event.set()  # Signal that this process is ready when port is shown
    stdout_task = asyncio.create_task(read_stream(process.stdout, color, label))
    stderr_task = asyncio.create_task(read_stream(process.stderr, Fore.RED, f"{label}-ERR"))
    try:
        await asyncio.gather(stdout_task, stderr_task)
        code = await process.wait()
        raise RuntimeError(f"{label} exited with code {code}; inspect its log above.")
    finally:
        # Interrupting this cell stops only the process group it started.
        try:
            os.killpg(process.pid, signal.SIGTERM)
            await asyncio.wait_for(process.wait(), timeout=15)
        except asyncio.TimeoutError:
            os.killpg(process.pid, signal.SIGKILL)
            await process.wait()
        except ProcessLookupError:
            pass
        for task in (stdout_task, stderr_task):
            task.cancel()

async def run_vlm(port=9000, **kwargs):
    await run_cmd(f"""
    vllm serve "nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16"
    --max_model_len 8192
    --gpu-memory-utilization {budget["vision_fraction"]}
    --limit-mm-per-prompt '{{"image":1,"video":0}}'
    --enforce-eager
    --port {port}
    --max-num-seqs 2
    --trust-remote-code
    --revision ca9543b126e8bf3176916d3d305ccc415f89fd4d
    --code-revision ca9543b126e8bf3176916d3d305ccc415f89fd4d
    """, label="VLM", color=Fore.BLUE, **kwargs) 

async def run_sdxl(port=9000, **kwargs):
    await run_cmd(
        f"uvicorn sdxl_app:app --host 0.0.0.0 --port {port} --log-level info",
        label="FLUX", color=Fore.CYAN, **kwargs) 

async def run_router(port=9000, **kwargs):
    await run_cmd(f"uvicorn router_app:app --host 0.0.0.0 --port {port} --log-level info",
    label="ROUTER", color=Fore.MAGENTA, **kwargs) 
    
# Create event objects to coordinate the start of each process
events = [None, None]
def get_ev_pair(events=events):
    events[:2] = [events[1], asyncio.Event()]
    return {"start_event": events[0], "end_event": events[1]}

print("Starting Processes")
tasks = [asyncio.create_task(job) for job in (
    run_vlm     (9002, **get_ev_pair()),
    run_sdxl    (9003, **get_ev_pair()),
    run_router  (9004, **get_ev_pair()),
)]
try:
    await asyncio.gather(*tasks)
finally:
    for task in tasks:
        task.cancel()
    await asyncio.gather(*tasks, return_exceptions=True)

Starting Processes

STARTING [VLM]
[VLM] INFO 10-01 15:25:28 [__init__.py:216] Automatically detected platform cuda.
[VLM] (APIServer pid=213) INFO 10-01 15:25:33 [api_server.py:1839] vLLM API server version 0.11.0
[VLM] (APIServer pid=213) INFO 10-01 15:25:33 [utils.py:233] non-default args: {'model_tag': 'nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16', 'port': 9002, 'model': 'nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16', 'trust_remote_code': True, 'revision': 'ca9543b126e8bf3176916d3d305ccc415f89fd4d', 'code_revision': 'ca9543b126e8bf3176916d3d305ccc415f89fd4d', 'max_model_len': 8192, 'enforce_eager': True, 'gpu_memory_utilization': 0.35, 'limit_mm_per_prompt': {'image': 1, 'video': 0}, 'max_num_seqs': 2}
[VLM] (APIServer pid=213) INFO 10-01 15:25:47 [model.py:547] Resolved architecture: NemotronH_Nano_VL_V2
[VLM] (APIServer pid=213) INFO 10-01 15:25:47 [model.py:1510] Using max model len 8192
[VLM] (APIServer pid=213) INFO 10-01 15:25:47 [scheduler.py:205] Chunked prefill is enabled wit

<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>